In [0]:
df = spark.read.load("/Volumes/filestore/tables/data/products.csv",format="csv",header =True)
df.display()

In [0]:
delta_table_path = "/delta/product-delta"
df.write.format("delta").save(delta_table_path)

In [0]:
%sh
ls /dbfs/delta/products-delta

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

#Create a deltaTable object
deltaTable = DeltaTable.forPath(spark, delta_table_path)
#update the table (reduce price of product 771 bt 10%)
deltaTable.update(
    condition= "ProductID == 771",
    set = {"ListPrice" : "ListPrice * 0.9"})

    #viwe the updated date as dataframe
deltaTable.toDF().show(10)

In [0]:
new_df = spark.read.format("delta").load(delta_table_path)
new_df.show(10)

In [0]:
new_df = spark.read.format("delta").option("versionAsOf",0).load(delta_table_path)
new_df.show(10)

In [0]:
deltaTable.history(10).show(10,False,True)

In [0]:
spark.sql("CREATE DATABASE AdventureWorks")
spark.sql("CREATE TABLE AdventureWorks.ProductsExternal USING DELTA LOCATION '{0}'".format(delta_table_path))
spark.sql("DESCRIBE EXTENDED AdventureWorks.ProductsExternal").show(truncate=False)

In [0]:
%sql
USE AdventureWorks;
SELECT * FROM ProductsExternal

In [0]:
df.write.format("delta").saveAsTable("AdventureWorks.ProductsManaged")
spark.sql("DESCRIBE EXTENDED AdventureWorks.ProductsManaged").show(truncate=False)

In [0]:
%sql
USE AdventureWorks;
show tables

In [0]:
%sh
echo "External table"
ls/dbfs/delta/products-delta
echo
echo "Managed table"
ls/dbfs/use/hive/warehose/AdventureWorks.db/ProductsManaged

In [0]:
%sql
use adventureworks;
drop table if exists ProductsExternal;
drop table if exists ProductsManaged;
show table;

In [0]:
%sql
USE adventureworks;
create table Products
using delta
location"/delta/products-delta"

In [0]:
%sql
USE adventureworks;
select * from Products

In [0]:
spark.sql("OPTIMIZE Products")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enable","False")
spark.sql("VACUUM Products RETAIN 24 HOURS")